# Rocco Evaluation: Statistical Analysis

This notebook provides comprehensive statistical analysis of Rocco (automated AI evaluator) against 5 human expert evaluators across 5 published porous media datasets.

**Methods:**
- Krippendorff's alpha for inter-rater reliability
- Bayesian Cumulative Link Mixed Model (CLMM) for LLM leniency
- Many Facet Item Response Theory (MFIRT) for faceted analysis

**Data:** 5 datasets × 10 rubric items × 6 evaluators (Rocco + 5 humans) = 300 evaluations

## Setup: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import krippendorff
import pymc as pm
import pytensor.tensor as pt

# Configure plotting
mpl.rcParams["font.size"] = 12
mpl.rcParams["figure.dpi"] = 100
colormap = "viridis"

## Section 1: Load and Prepare Data

In [ ]:
# Load evaluation data
df = pd.read_excel("data/evaluation_results.xlsx", sheet_name="flat_file")
print(f"Loaded {len(df)} evaluation records")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Define score mapping: {0, 0.5, 1} → {0, 1, 2} for ordinal modeling
def rating_map(x):
    score_map = {0: 0, 0.5: 1, 1: 2}
    try:
        return score_map[float(x)]
    except KeyError:
        return np.nan

df["Rating"] = df["Score"].apply(rating_map)
print(f"Rating distribution (ordinal 0-2):")
print(df['Rating'].value_counts().sort_index())

# Create index maps for modeling
raters = sorted(df["Grader"].unique())
descs = sorted(df["Description"].unique())
items = sorted(df["Rubric Item"].unique())

rater_to_idx = {r: i for i, r in enumerate(raters)}
desc_to_idx = {d: i for i, d in enumerate(descs)}
item_to_idx = {it: i for i, it in enumerate(items)}

df["rater_idx"] = df["Grader"].map(rater_to_idx).astype(int)
df["desc_idx"] = df["Description"].map(desc_to_idx).astype(int)
df["item_idx"] = df["Rubric Item"].map(item_to_idx).astype(int)

LLM_NAME = "Rocco"
df["is_llm"] = (df["Grader"] == LLM_NAME).astype(int)

print(f"Raters: {len(raters)} ({', '.join(raters)})")
print(f"Descriptions: {len(descs)}")
print(f"Rubric Items: {len(items)}")

## Section 2: Data Exploration — Raw Score Distributions

In [ ]:
# Prepare data for plotting
score_order = [0, 0.5, 1]
df_plot = df.copy()
df_plot["Score"] = pd.Categorical(df_plot["Score"], categories=score_order, ordered=True)

# Relabel graders for readability
human_graders = sorted(g for g in df_plot["Grader"].unique() if g != "Rocco")
human_map = {grader: f"Evaluator {i+1}" for i, grader in enumerate(human_graders)}
df_plot["Grader_label"] = df_plot["Grader"].replace(human_map).replace({"Rocco": "Rocco AI"})
grader_order = ["Rocco AI"] + [f"Evaluator {i+1}" for i in range(len(human_graders))]
df_plot["Grader_label"] = pd.Categorical(df_plot["Grader_label"], categories=grader_order, ordered=True)

print("Data prepared for plotting.")

In [ ]:
# Plot 1: Overall score distribution by grader
prop_df = df_plot.groupby(["Grader_label", "Score"]).size().reset_index(name="count")
prop_df["proportion"] = prop_df.groupby("Grader_label")["count"].transform(lambda x: x / x.sum())
plot_df = prop_df.pivot(index="Grader_label", columns="Score", values="proportion").fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
plot_df.plot(kind="bar", stacked=True, ax=ax, colormap=colormap)
ax.set_ylabel("Proportion of ratings")
ax.set_xlabel("Grader")
ax.set_title("Distribution of Scores by Grader")
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(title="Score", labels=["0 (Poor)", "0.5 (Adequate)", "1 (Good)"], bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Score distribution faceted by description
prop_desc_df = df_plot.groupby(["Description", "Grader_label", "Score"]).size().reset_index(name="count")
prop_desc_df["proportion"] = prop_desc_df.groupby(["Description", "Grader_label"])["count"].transform(
    lambda x: x / x.sum()
)

descriptions = sorted(df_plot["Description"].unique())
fig, axes = plt.subplots(nrows=1, ncols=len(descriptions), figsize=(4 * len(descriptions), 5), sharey=True)

for i, (ax, desc) in enumerate(zip(axes, descriptions), start=1):
    sub = prop_desc_df[prop_desc_df["Description"] == desc].copy()
    grader_order_facet = ["Rocco AI"] + [g for g in sorted(sub["Grader_label"].unique()) if g != "Rocco AI"]
    sub["Grader_label"] = pd.Categorical(sub["Grader_label"], categories=grader_order_facet, ordered=True)
    plot_df = sub.pivot(index="Grader_label", columns="Score", values="proportion").fillna(0)

    plot_df.plot(kind="bar", stacked=True, ax=ax, legend=False, colormap=colormap)
    ax.set_title(f"{desc}")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

axes[0].set_ylabel("Proportion of ratings")

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, ["0 (Poor)", "0.5 (Adequate)", "1 (Good)"], title="Score", bbox_to_anchor=(1.02, 0.9))
fig.suptitle("Score Distributions by Description", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Raw score distributions by rubric item (humans pooled vs Rocco AI)
df_plot["Grader_group"] = df_plot["Grader_label"].apply(lambda x: "Rocco AI" if x == "Rocco AI" else "Humans")

prop_item_df = df_plot.groupby(["Rubric Item", "Grader_group", "Score"]).size().reset_index(name="count")
prop_item_df["proportion"] = prop_item_df.groupby(["Rubric Item", "Grader_group"])["count"].transform(
    lambda x: x / x.sum()
)

rubric_items = sorted(df_plot["Rubric Item"].unique())
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for ax, item in zip(axes, rubric_items):
    sub = prop_item_df[prop_item_df["Rubric Item"] == item]
    plot_df = sub.pivot(index="Grader_group", columns="Score", values="proportion").fillna(0)
    plot_df.plot(kind="bar", stacked=True, ax=ax, legend=False, colormap=colormap)
    ax.set_title(f"Item {item}")
    ax.set_xlabel("")
    ax.set_ylim(0, 1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

axes[0].set_ylabel("Proportion of ratings")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, ["0 (Poor)", "0.5 (Adequate)", "1 (Good)"], title="Score", bbox_to_anchor=(1.02, 0.9))
fig.suptitle("Score Distributions by Rubric Item (Humans pooled vs Rocco AI)", y=1.02)
plt.tight_layout()
plt.show()

## Section 3: Inter-Rater Reliability (Krippendorff's Alpha)

**What it measures:** Krippendorff's alpha quantifies how consistently raters agree on their evaluations, accounting for the possibility that some agreement occurs by chance. It ranges from -∞ to 1, where 1 = perfect agreement and 0 = agreement at chance levels.

We compute alpha for:
1. All raters together (including Rocco)
2. Human raters only
3. The difference between these (Δα) to assess Rocco's impact on agreement

### Building the Reliability Matrix

A **reliability matrix** is a table where each row is a rater and each column represents a unique pair of dataset and rubric item. We fill this table with each evaluator's score (0, 1, or 2) for that combination. This structure lets us directly compare how consistently raters scored the same items—the key input for calculating inter-rater agreement.

In [ ]:
# Create reliability matrix: rows=raters, cols=description-item pairs
cells = [(d, i) for d in descs for i in items]
mat = np.full((len(raters), len(cells)), np.nan, dtype=float)

cell_idx = {cell: idx for idx, cell in enumerate(cells)}
rater_idx_dict = {rater: idx for idx, rater in enumerate(raters)}

for _, row in df.iterrows():
    rater = row["Grader"]
    desc = row["Description"]
    item = row["Rubric Item"]
    rating = row["Rating"]

    if pd.notna(rating):
        rr = rater_idx_dict[rater]
        cc = cell_idx[(desc, item)]
        mat[rr, cc] = rating

print(f"Reliability matrix shape: {mat.shape}")
print(f"Data completeness: {100 * (1 - np.isnan(mat).sum() / mat.size):.1f}%")

### Computing Krippendorff's Alpha

**Krippendorff's alpha (α)** is a number from 0 to 1 that measures agreement among raters. 

An α near **1** means raters agree closely; **0** means agreement is no better than random chance.

We compute α three ways: (1) including all raters including Rocco, (2) human raters only, and (3) the difference between (1) and (2), which shows whether Rocco adds to or detracts from overall agreement.

In [ ]:
# Compute Krippendorff's alpha
alpha_all = krippendorff.alpha(reliability_data=mat, level_of_measurement="ordinal")

# Compute for humans only (excluding Rocco)
raters_human = [r for r in raters if r != LLM_NAME]
rows_human = [rater_idx_dict[r] for r in raters_human]
mat_human = mat[rows_human, :]

alpha_human = krippendorff.alpha(reliability_data=mat_human, level_of_measurement="ordinal")

print(f"\nKrippendorff's Alpha (point estimates):")
print(f"  All raters (including Rocco):  α = {alpha_all:.4f}")
print(f"  Humans only:                   α = {alpha_human:.4f}")

### Bootstrap Confidence Intervals for Alpha

Instead of reporting a single alpha value, we use **bootstrapping**: repeatedly resampling our data (with replacement) and recalculating alpha each time. This gives us a distribution of likely alpha values. We account for uncertainty due to finite sample size by computing the 95% confidence interval (95% CI).

In [ ]:
# Bootstrap confidence intervals
def bootstrap_alpha_columns(mat, B=1000, seed=42, level="ordinal"):
    rng = np.random.default_rng(seed)
    n_cols = mat.shape[1]
    alphas = []

    for _ in range(B):
        cols = rng.integers(0, n_cols, size=n_cols)
        mat_b = mat[:, cols]
        try:
            alpha = krippendorff.alpha(reliability_data=mat_b, level_of_measurement=level)
            alphas.append(alpha)
        except Exception:
            pass
    return np.array(alphas)

print("Computing bootstrap CIs (10,000 resamples, this may take a minute)...")
alphas_all_boot = bootstrap_alpha_columns(mat, B=10000, seed=42)
alphas_human_boot = bootstrap_alpha_columns(mat_human, B=10000, seed=42)

ci_all = np.quantile(alphas_all_boot, [0.025, 0.975])
ci_human = np.quantile(alphas_human_boot, [0.025, 0.975])

print(f"\nKrippendorff's Alpha (with bootstrap 95% CIs):")
print(f"  All raters:   α = {alpha_all:.4f}, 95% CI [{ci_all[0]:.4f}, {ci_all[1]:.4f}]")
print(f"  Humans only:  α = {alpha_human:.4f}, 95% CI [{ci_human[0]:.4f}, {ci_human[1]:.4f}]")

### Calculating the Impact of Rocco on Overall Agreement (Δα)

The difference **Δα = α(all raters) − α(humans only)** tells us whether adding Rocco's scores improves, worsens, or leaves unchanged the overall rater agreement. 

A positive Δα suggests Rocco's scores align well with the human consensus; negative or near-zero suggests Rocco operates independently. We bootstrap this difference too, so we get a full distribution and can quantify uncertainty around the true impact.

In [ ]:
# Bootstrap difference in alpha
def bootstrap_delta_alpha(mat_all, mat_hum, B=5000, seed=123, level="ordinal"):
    rng = np.random.default_rng(seed)
    n_cols = mat_all.shape[1]
    deltas = []

    for _ in range(B):
        cols = rng.integers(0, n_cols, size=n_cols)
        mat_a = mat_all[:, cols]
        mat_h = mat_hum[:, cols]

        try:
            a_all = krippendorff.alpha(reliability_data=mat_a, level_of_measurement=level)
            a_hum = krippendorff.alpha(reliability_data=mat_h, level_of_measurement=level)
        except Exception:
            continue

        if np.isfinite(a_all) and np.isfinite(a_hum):
            deltas.append(a_all - a_hum)

    return np.array(deltas)

print("Computing bootstrap Δα (5,000 resamples, this may take a minute)...")
deltas = bootstrap_delta_alpha(mat, mat_human, B=5000, seed=123)

mean_delta = float(np.mean(deltas))
ci_delta = np.quantile(deltas, [0.025, 0.975])
pr_positive = float(np.mean(deltas > 0))

print(f"\nImpact of Including Rocco on Alpha:")
print(f"  Δα (all - humans) = {mean_delta:.4f}")
print(f"  95% CI: [{ci_delta[0]:.4f}, {ci_delta[1]:.4f}]")
print(f"  P(Δα > 0) = {pr_positive:.4f}")

### Interpreting the Alpha Distributions

The left histogram shows the bootstrap distribution of α values with and without Rocco. The dashed lines mark the expected α value for each group. The right histogram shows the distribution of Δα differences. 

In [ ]:
# Visualize alpha distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alpha distributions
ax = axes[0]
ax.hist(alphas_all_boot, bins=30, alpha=0.6, label="Human + Rocco", color="steelblue")
ax.hist(alphas_human_boot, bins=30, alpha=0.6, label="Humans only", color="coral")
ax.axvline(alpha_all, color="steelblue", linestyle="--", linewidth=2, label=f"α (all) = {alpha_all:.3f}")
ax.axvline(alpha_human, color="coral", linestyle="--", linewidth=2, label=f"α (hum) = {alpha_human:.3f}")
ax.set_xlabel("Krippendorff's α")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of Krippendorff's Alpha")
ax.legend()
ax.grid(alpha=0.3)

# Delta alpha distribution
ax = axes[1]
ax.hist(deltas, bins=30, alpha=0.7, color="mediumpurple")
ax.axvline(mean_delta, color="darkviolet", linestyle="--", linewidth=2, label=f"Mean Δα = {mean_delta:.4f}")
ax.axvline(0, color="gray", linestyle=":", linewidth=1)
ax.set_xlabel("Δα = α(all) - α(humans)")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of Alpha Difference")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Section 4: Bayesian Cumulative Link Mixed Model (CLMM)

**What it measures:** The CLMM estimates whether Rocco is systematically more or less lenient than human evaluators. It models the evaluation scores (0, 0.5, 1) as a function of systematic random effects between rater leniency, description quality, and rubric-item difficulty.

**Key result:** The leniency contrast is the difference in rater intercepts (Rocco vs. human average). This is the primary statistical approach reported in the associated IJDC conference paper (Chang et al., 2026).

### Preparing Data for the Bayesian CLMM Model

Before fitting the model, we convert our scores into ordinal indices (0, 1, 2) and create index arrays that track which rater, description, and rubric item each observation belongs to. These indices allow the Bayesian model to separately estimate the contribution of each rater, description, and item to the final scores.

In [ ]:
# Prepare data for CLMM
y = df["Rating"].to_numpy()  # 0/1/2 ratings
rater_idx_arr = df["rater_idx"].to_numpy()  # 0..R-1
desc_idx_arr = df["desc_idx"].to_numpy()  # 0..D-1
item_idx_arr = df["item_idx"].to_numpy()  # 0..I-1

R = int(rater_idx_arr.max() + 1)
D = int(desc_idx_arr.max() + 1)
I = int(item_idx_arr.max() + 1)
llm_idx = rater_to_idx[LLM_NAME]
human_ids = [r for r in range(R) if r != llm_idx]

print(f"CLMM data: N={len(y)}, R={R} raters, D={D} descriptions, I={I} items")

### Fitting the Bayesian CLMM Model

The **Cumulative Link Mixed Model (CLMM)** is a Bayesian approach that models ordinal outcomes (0, 0.5, 1) as arising from a latent continuous variable that captures each evaluator's overall strictness, description quality rubric item difficulty. The model uses "sampling" to draw thousands of plausible parameter values from the posterior distribution, accounting for uncertainty.

In [ ]:
# Fit Bayesian CLMM model with item-specific LLM interaction
print("Fitting Bayesian CLMM with item-specific LLM effects (this may take a few minutes)...")

with pm.Model() as m_clmm:
    # Global location
    mu = pm.Normal("mu", 0, 1)

    # Non-centered random effects: rater, description, item
    sigma_r = pm.HalfNormal("sigma_r", 0.25)
    z_r = pm.Normal("z_r", 0, 1, shape=R)
    u_r = pm.Deterministic("u_r", sigma_r * z_r)

    sigma_d = pm.HalfNormal("sigma_d", 0.25)
    z_d = pm.Normal("z_d", 0, 1, shape=D)
    v_d = pm.Deterministic("v_d", sigma_d * z_d)

    sigma_i = pm.HalfNormal("sigma_i", 0.25)
    z_i = pm.Normal("z_i", 0, 1, shape=I)
    w_i = pm.Deterministic("w_i", sigma_i * z_i)

    # LLM-by-item interaction: item-specific LLM shift (hierarchical shrinkage)
    sigma_g = pm.HalfNormal("sigma_g", 0.25)
    z_g = pm.Normal("z_g", 0, 1, shape=I)
    gamma_i = pm.Deterministic("gamma_i", sigma_g * z_g)

    # Ordered global cutpoints
    tau0 = pm.Normal("tau0", 0, 0.5)
    d_tau = pm.HalfNormal("d_tau", 0.5)
    tau1 = pm.Deterministic("tau1", tau0 + d_tau)
    cutpoints = pt.stack([tau0, tau1])

    # Indicator for LLM rows
    is_llm = (rater_idx_arr == llm_idx).astype(float)

    # Linear predictor with item-specific LLM effect
    eta = mu + u_r[rater_idx_arr] + v_d[desc_idx_arr] + w_i[item_idx_arr] + gamma_i[item_idx_arr] * is_llm

    # Likelihood: ordered logistic
    y_obs = pm.OrderedLogistic("y_obs", eta=eta, cutpoints=cutpoints, observed=y)

    # Sample
    trace_clmm = pm.sample(
        3000, tune=3000, target_accept=0.99, max_treedepth=15, random_seed=42, progressbar=True
    )

print("\n✓ CLMM fitting complete")

### Interpreting the CLMM Leniency Contrast

Here, we compute the global leniency contrast. This gives an idea of whether Rocco is harsher or more lenient than the average human evaluator.

Negative contrast values indicate that Rocco is systematically harsher than the human evaluators, positive contrast values indicate greater leniency, and values near zero suggest no systematic difference. 

To address uncertainty due to sample sizes, we also present the 95% credible interval (CrI). 

In [ ]:
# Extract and summarize LLM leniency contrast
posterior = trace_clmm.posterior
u_post = posterior["u_r"].values

llm_u = u_post[..., llm_idx]
hum_u = u_post[..., human_ids]
hum_mean = hum_u.mean(axis=-1)  # average over human raters

# Contrast: LLM vs human average (leniency scale)
diff = (llm_u - hum_mean).flatten()

print("\n" + "="*70)
print("CLMM Results: LLM Leniency Contrast")
print("="*70)
print(f"  Median: {np.median(diff):.4f}")
print(f"  95% CrI: [{np.quantile(diff, 0.025):.4f}, {np.quantile(diff, 0.975):.4f}]")
print(f"  P(Rocco more lenient): {np.mean(diff > 0):.4f}")

### CLMM Leniency Contrast per Rubric Item

We can similarly compute the CLMM leniency contrast individually for each rubric item. Leniency contrast values that diverge from zero indicate criteria where Rocco tends to disagree with the human evaluators. This helps identify rubric items that need additional attention and clarification. 

In [ ]:
# Extract and summarize LLM leniency contrast (global + per-item)
posterior = trace_clmm.posterior
u_post = posterior["u_r"].values
g_post = posterior["gamma_i"].values

llm_u = u_post[..., llm_idx]
hum_u = u_post[..., human_ids]
hum_mean = hum_u.mean(axis=-1)

# Global contrast: LLM vs human average (leniency scale)
llm_u = llm_u[..., None]
hum_mean = hum_mean[..., None]
diff_global = (llm_u - hum_mean) + g_post
print("\n" + "="*70)
print("CLMM Results: Global LLM Leniency Contrast")
print("="*70)
print(f"  Median: {np.median(diff_global):.4f}")
print(f"  95% CrI: [{np.quantile(diff_global, 0.025):.4f}, {np.quantile(diff_global, 0.975):.4f}]")
print(f"  P(Rocco more lenient): {np.mean(diff_global > 0):.4f}")

# Per-item contrasts using gamma_i
print("\n" + "="*70)
print("Per-Item LLM Leniency Contrasts (gamma_i)")
print("="*70)
per_item_gamma = []
for i in range(I):
    gamma_i_samples = g_post[..., i].flatten()  # (chains*draws,)
    median_val = np.median(gamma_i_samples)
    ci_low = np.quantile(gamma_i_samples, 0.025)
    ci_high = np.quantile(gamma_i_samples, 0.975)
    p_positive = np.mean(gamma_i_samples > 0)

    per_item_gamma.append({
        "Item": i + 1,
        "Median": median_val,
        "Lower95": ci_low,
        "Upper95": ci_high,
        "P_positive": p_positive
    })

    print(f"Item {i+1:2d}: Δ_LLM median={median_val:7.4f}, 95% CrI [{ci_low:7.4f}, {ci_high:7.4f}], P(pos)={p_positive:.3f}")

gamma_results_df = pd.DataFrame(per_item_gamma)

### Reading the Forest Plot

A **forest plot** displays a point estimate (the dot) and its 95% credible interval (the error bars). 

The vertical dashed line at zero is the reference (indicating no difference between Rocco and the human evaluators). If the interval crosses zero, the evidence for a difference is weak; if it lies entirely to the right or left, the evidence is stronger.

In [ ]:
### Forest Plot: Per-Item LLM Leniency
# Prepare data for forest plot
delta_df = gamma_results_df.copy()
delta_df['Rubric_Item'] = 'Item ' + delta_df['Item'].astype(str)

# Sort by absolute value of median (largest effects first)
# delta_df['AbsMedian'] = np.abs(delta_df['Median'])
# delta_df['CrI_width'] = delta_df['Upper95'] - delta_df['Lower95']
delta_df = delta_df.sort_values('Median', ascending=True)

# Plot forest plot
fig, ax = plt.subplots(figsize=(10, len(delta_df) * 0.5 + 1))

y_pos = range(len(delta_df))

# Plot error bars (95% CrI)
ax.errorbar(
    delta_df["Median"],
    y_pos,
    xerr=[delta_df["Median"] - delta_df["Lower95"], delta_df["Upper95"] - delta_df["Median"]],
    fmt='o',
    color='navy',
    ecolor='skyblue',
    elinewidth=2,
    capsize=4,
    markersize=7
)

# Add zero line for reference
ax.axvline(x=0, color='gray', linestyle='--', linewidth=1.5)

# Y-axis labels
ax.set_yticks(y_pos)
ax.set_yticklabels(delta_df["Rubric_Item"])
ax.set_xlabel("$\\Delta_{LLM}$ (Rocco − Human Average)", fontsize=12)
ax.set_ylabel("Rubric Item", fontsize=12)
ax.set_title("Per-Item LLM Leniency: Item-Specific Rocco Effects", fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3, linestyle=':')

plt.gca().invert_yaxis()  # largest effect on top
plt.tight_layout()
plt.show()

## Section 5: Many Facet Item Response Theory (MFIRT)

**What it measures:** MFIRT decomposes the ordinal scores into separate facets:
- **Description quality (θ)** — How good is each dataset's original description
- **Item difficulty (β)** — How hard is each rubric criterion to satisfy
- **Rater severity (φ)** — How strict or lenient is each evaluator

This helps separate *what* Rocco is measuring (description quality) from *how* it measures (rater severity). We also compute expected rubric scores for each description as rated by a neutral evaluator.

### Fitting the MFIRT Model

**Many Facet Item Response Theory (MFIRT)** goes deeper than the CLMM by decomposing scores into separate, interpretable facets:
- **θ (theta)**: latent quality of each description (how intrinsically good it is)
- **β (beta)**: difficulty of each rubric item (how hard it is to score well on that criterion)
- **φ (phi)**: severity/strictness of each rater (how harsh or lenient they are)

This separation lets us answer: "Is description 1 truly better, or did rater 1 just score it more favorably?"

In [ ]:
# Helper function for expected rubric score
def expected_rubric(eta, t0, t1):
    """Compute expected ordinal outcome from latent location and thresholds."""
    p_le0 = 1.0 / (1.0 + np.exp(-(t0 - eta)))
    p_le1 = 1.0 / (1.0 + np.exp(-(t1 - eta)))
    p0, p1, p2 = p_le0, (p_le1 - p_le0), (1.0 - p_le1)
    Ey = 0 * p0 + 1 * p1 + 2 * p2
    return Ey / 2.0  # scale to {0, 0.5, 1}

print("Fitting MFIRT model (this may take a few minutes)...\n")

with pm.Model() as m_mfirt:
    # Facet SDs
    sigma_theta = pm.HalfNormal("sigma_theta", 0.5)
    sigma_beta = pm.HalfNormal("sigma_beta", 0.5)
    sigma_phi = pm.HalfNormal("sigma_phi", 0.5)

    # Non-centered facet effects
    z_theta = pm.Normal("z_theta", 0, 1, shape=D)
    z_beta = pm.Normal("z_beta", 0, 1, shape=I)
    z_phi = pm.Normal("z_phi", 0, 1, shape=R)

    theta = pm.Deterministic("theta", sigma_theta * z_theta)
    beta = pm.Deterministic("beta", sigma_beta * z_beta)
    phi = pm.Deterministic("phi", sigma_phi * z_phi)

    # Center facets (sum-to-zero constraints)
    theta_c = pm.Deterministic("theta_c", theta - pt.mean(theta))
    beta_c = pm.Deterministic("beta_c", beta - pt.mean(beta))
    phi_c = pm.Deterministic("phi_c", phi - pt.mean(phi))

    # Item-specific ordered thresholds
    tau0_i = pm.Normal("tau0_i", 0, 1, shape=I)
    d_tau_i = pm.HalfNormal("d_tau_i", 1.0, shape=I)
    tau1_i = pm.Deterministic("tau1_i", tau0_i + d_tau_i)

    # Assemble cutpoints and latent location
    cutpoints = pt.stack([tau0_i[item_idx_arr], tau1_i[item_idx_arr]], axis=1)
    loc = theta_c[desc_idx_arr] - beta_c[item_idx_arr] - phi_c[rater_idx_arr]

    # Likelihood
    y_obs = pm.OrderedLogistic("y_obs", eta=loc, cutpoints=cutpoints, observed=y)

    # Sample
    trace_mfirt = pm.sample(
        3000, tune=3000, target_accept=0.99, max_treedepth=15, random_seed=42, progressbar=True
    )

print("\n✓ MFIRT fitting complete")

### Interpreting Rater Severity (φ)

In MFIRT, **severity (φ)** captures each evaluator's strict or lenient tendency, isolated from the quality of what they're evaluating. 

A positive difference (Rocco − human average) means Rocco is stricter; negative means more lenient. The 95% credible interval tells us the range of plausible differences.

In [ ]:
# Extract and summarize MFIRT results
phi_post = trace_mfirt.posterior["phi"].values  # (chains, draws, R)
phi_llm = phi_post[..., llm_idx]
phi_hums = phi_post[..., human_ids].mean(axis=-1)

diff_phi = (phi_llm - phi_hums).flatten()

print("\n" + "="*70)
print("MFIRT Results: Rater Severity")
print("="*70)
print("\nRocco Severity vs Human Average:")
print(f"  Median difference (φ_Rocco - φ_Human): {np.median(diff_phi):.4f}")
print(f"  95% CrI: [{np.quantile(diff_phi, 0.025):.4f}, {np.quantile(diff_phi, 0.975):.4f}]")
print(f"  P(Rocco stricter): {np.mean(diff_phi > 0):.4f}")
print("\nInterpretation (positive = stricter; negative = more lenient):")

### Understanding Item Difficulty and Threshold Width

**Difficulty (β)** reflects how hard a rubric item is to achieve. Positive β means the item is easy to score well on (raters tend to assign higher scores); negative β means it's hard. 

**Threshold width (Δτ)** is the gap between the two ordinal (scoring) cutpoints—larger gaps mean wider separation between score categories for that item, while smaller gaps mean the categories are squeezed together (hard to distinguish them in practice).

In [ ]:
# Item difficulty and spread
beta = trace_mfirt.posterior["beta_c"].values.reshape(-1, I)
tau0 = trace_mfirt.posterior["tau0_i"].values.reshape(-1, I)
tau1 = trace_mfirt.posterior["tau1_i"].values.reshape(-1, I)
dtau = tau1 - tau0

print("\n" + "="*70)
print("Item Difficulty and Threshold Width")
print("="*70)
for i in range(I):
    b = beta[:, i]
    spread = dtau[:, i]
    item_name = f"Item {items[i]}"
    print(f"\n{item_name}:")
    print(f"  Difficulty (β):      {np.median(b):7.4f}, 95% CrI [{np.quantile(b, 0.025):7.4f}, {np.quantile(b, 0.975):7.4f}]")
    print(f"  Threshold width (Δτ): {np.median(spread):7.4f}, 95% CrI [{np.quantile(spread, 0.025):7.4f}, {np.quantile(spread, 0.975):7.4f}]")

### Interpreting Description Quality Estimates

The **latent trait θ (theta)** estimated for each description is a model-based estimate of its inherent quality, free from rater bias and item difficulty effects. A higher θ means the description is stronger overall; lower θ means weaker. These estimates let you compare descriptions on a common scale—unlike raw scores, which are confounded by who rated them and which items were harder.

In [ ]:
# Description quality estimates
theta = trace_mfirt.posterior["theta_c"].values.reshape(-1, D)

print("\n" + "="*70)
print("Description Quality Estimates (latent trait θ)")
print("="*70)
for d in range(D):
    t = theta[:, d]
    desc_name = descs[d]
    print(f"{desc_name}: {np.median(t):7.4f}, 95% CrI [{np.quantile(t, 0.025):7.4f}, {np.quantile(t, 0.975):7.4f}]")

### Computing Expected Rubric Scores from a Neutral Evaluator

This step synthesizes the MFIRT estimates: for each description, we compute what its expected total score (out of 10) would be if rated by a hypothetical **neutral evaluator** (not too strict, not too lenient). This "neutral" score isolates the description's true quality from rater and item effects, giving a fair comparison point. Descriptions with higher neutral scores are genuinely stronger; those with lower scores have more intrinsic room for improvement.

In [ ]:
# Expected total rubric scores from neutral grader
theta_s = trace_mfirt.posterior["theta_c"].values.reshape(-1, D)
beta_s = trace_mfirt.posterior["beta_c"].values.reshape(-1, I)
tau0_s = trace_mfirt.posterior["tau0_i"].values.reshape(-1, I)
tau1_s = trace_mfirt.posterior["tau1_i"].values.reshape(-1, I)

S = theta_s.shape[0]
totals = np.zeros((S, D))

for s in range(S):
    for d in range(D):
        eta_di = theta_s[s, d] - beta_s[s, :]  # (I,)
        totals[s, d] = np.sum(expected_rubric(eta_di, tau0_s[s, :], tau1_s[s, :]))

print("\n" + "="*70)
print("Expected Total Rubric Score (from neutral evaluator, on 0-10 scale)")
print("="*70)
for d in range(D):
    q = np.quantile(totals[:, d], [0.025, 0.5, 0.975])
    desc_name = descs[d]
    print(f"{desc_name}: {q[1]:.2f} / 10, 95% CrI [{q[0]:.2f}, {q[2]:.2f}]")